# Ambient Log-Gram Alpha Scan

Runs the checkpoint-only max-mix/pure campaign through the canonical GPU circuit runner. The hard exterior preparation is replayable but excluded from the cocycle; Choi tracking is disabled. Pilot and production are both opt-in and independently checksum-resumable.

## 1. Mount Drive and inspect the immutable contract


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
from pathlib import Path
import importlib.util, json, subprocess, sys
MYDRIVE = Path('/content/drive/MyDrive')
CAMPAIGN_ROOT = MYDRIVE / 'final_production_new_designs'
BUNDLE_ROOT = CAMPAIGN_ROOT / '07_log_gram_alpha_scan'
if not (BUNDLE_ROOT / 'run_bundle.py').is_file():
    raise FileNotFoundError(f'Missing uploaded bundle: {BUNDLE_ROOT}')
config = json.loads((BUNDLE_ROOT / 'production_config.json').read_text())
runner_path = BUNDLE_ROOT / 'run_bundle.py'
runner_spec = importlib.util.spec_from_file_location('_colab_log_gram_runner', runner_path)
if runner_spec is None or runner_spec.loader is None:
    raise ImportError(f'Cannot load runner: {runner_path}')
runner_module = importlib.util.module_from_spec(runner_spec)
sys.modules[runner_spec.name] = runner_module
runner_spec.loader.exec_module(runner_module)
print(json.dumps(config, indent=2))


## 2. Verify the A100 runtime


In [ ]:
RUN_A100_PREFLIGHT = True
if RUN_A100_PREFLIGHT:
    subprocess.run([sys.executable, '-u', str(BUNDLE_ROOT / 'run_bundle.py'), 'preflight'], check=True)


## 3. Optional pilot

The 12 pilot cases use two trajectories each and replay every saved record before accepting the shard. The pilot is disabled by default.


In [ ]:
PILOT_ROOT = MYDRIVE / 'classA_pilot_outputs' / config['campaign_revision']
PILOT_CASE_ID = None
RUN_PILOT = False
pilot_argv = ['pilot', '--output-root', str(PILOT_ROOT), '--replay-check']
if PILOT_CASE_ID is not None:
    pilot_argv += ['--case-id', PILOT_CASE_ID]
pilot_command = [sys.executable, '-u', str(runner_path), *pilot_argv]
print('[pilot]', ' '.join(pilot_command))
if RUN_PILOT:
    returncode = runner_module.main(pilot_argv)
    if returncode not in (None, 0):
        raise RuntimeError(f'Runner returned nonzero status {returncode}')


## 4. Resumable production queue

Set `RUN_PRODUCTION=True` only after the pilot manifests validate. Existing immutable shards are checksum-verified and skipped.


In [ ]:
PRODUCTION_ROOT = MYDRIVE / config['output_collection']
CASE_ID = None       # exact case ID, or None for all 220 cases
SHARD_INDEX = None   # 0..4, or None for all five shards
RUN_PRODUCTION = False
production_argv = ['production', '--output-root', str(PRODUCTION_ROOT)]
if CASE_ID is not None:
    production_argv += ['--case-id', CASE_ID]
if SHARD_INDEX is not None:
    production_argv += ['--shard-index', str(SHARD_INDEX)]
production_command = [sys.executable, '-u', str(runner_path), *production_argv]
print('[production]', ' '.join(production_command))
if RUN_PRODUCTION:
    returncode = runner_module.main(production_argv)
    if returncode not in (None, 0):
        raise RuntimeError(f'Runner returned nonzero status {returncode}')


## Final. Disconnect this completed Colab runtime


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
